In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor
import shap
from sklearn.datasets import load_diabetes
from sklearn.metrics import mean_absolute_error, mean_squared_error
import os
from datetime import datetime
import sys


# =============================================================================
# CONFIGURATION
# =============================================================================

OUTPUT_DIR = r"your output dir"
os.makedirs(OUTPUT_DIR, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
LOG_FILE = os.path.join(OUTPUT_DIR, f"lri_sigma_analysis_{timestamp}.txt")


# =============================================================================
# LOGGER
# =============================================================================

class Logger:
    """Classe pour écrire dans console et fichier"""
    def __init__(self, filename):
        self.terminal = sys.stdout
        self.log = open(filename, 'w', encoding='utf-8')

    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)
        self.log.flush()

    def flush(self):
        self.terminal.flush()
        self.log.flush()

    def close(self):
        self.log.close()


In [2]:
# =============================================================================
# SECTION 1: CHARGEMENT DU DATASET DIABETES ET PRÉPARATION DES DONNÉES
# =============================================================================

def load_diabetes_dataset():
    """Charge le dataset Diabetes depuis scikit-learn"""
    diabetes = load_diabetes(as_frame=True)
    df = diabetes.data.copy()
    df['target'] = diabetes.target
    return df


def introduce_missing_values(df, missing_rate, random_state=42):
    """
    Introduit des valeurs manquantes aléatoirement (MCAR)
    EXCLUSION: Ne pas introduire de valeurs manquantes dans 'target'

    Returns:
        tuple: (df_missing, mask_missing)
    """
    df_missing = df.copy()
    np.random.seed(random_state)

    # Identifier les colonnes features (exclure 'target')
    feature_cols = [col for col in df.columns if col != 'target']
    feature_indices = [df.columns.get_loc(col) for col in feature_cols]

    # Calculer le nombre total de cellules dans les features uniquement
    n_total_features = df.shape[0] * len(feature_cols)
    n_missing = int(n_total_features * missing_rate)

    # Générer tous les indices possibles (uniquement pour les features)
    indices = [(i, j) for i in range(df.shape[0]) for j in feature_indices]
    indices_missing = np.random.choice(len(indices), n_missing, replace=False)

    # Remplacer par NaN
    for idx in indices_missing:
        i, j = indices[idx]
        df_missing.iat[i, j] = np.nan

    mask_missing = df_missing.isnull()

    return df_missing, mask_missing


# =============================================================================
# SECTION 2: IMPUTATION AVEC EXTRACTION DE σᵢ
# =============================================================================

def impute_with_uncertainty_extraction(df_missing, df_original, mask_missing):
    """
    🎯 CŒUR DE L'INNOVATION

    Effectue l'imputation ET extrait σᵢ pour chaque valeur imputée.

    Pour chaque variable à imputer:
    1. Entraîner un RandomForestRegressor
    2. Récupérer les prédictions de TOUS les arbres individuellement
    3. Calculer σᵢ = std(prédictions des arbres)

    Returns:
        tuple: (df_imputed, uncertainty_dict, imputation_metrics)

        uncertainty_dict: {(row_idx, col_name): σᵢ}
    """
    print("\n" + "="*80)
    print("IMPUTATION AVEC EXTRACTION DE L'INCERTITUDE σᵢ")
    print("="*80)

    df_imputed = df_missing.copy()
    uncertainty_dict = {}  # Stocke σᵢ pour chaque cellule imputée

    feature_cols = [col for col in df_missing.columns if col != 'target']

    # 🔧 CORRECTION: Calculer les moyennes une seule fois au début
    col_means = {col: df_imputed[col].mean() for col in feature_cols}

    for target_col in feature_cols:
        # Vérifier s'il y a des valeurs manquantes
        if df_missing[target_col].isna().sum() == 0:
            continue

        # Préparation des données
        missing_mask = df_missing[target_col].isna()
        predictor_cols = [c for c in feature_cols if c != target_col]

        # Données d'entraînement (valeurs observées)
        train_mask = ~missing_mask
        X_train = df_imputed.loc[train_mask, predictor_cols].copy()

        # 🔧 CORRECTION: Remplir les NaN avec les moyennes
        for col in predictor_cols:
            X_train[col] = X_train[col].fillna(col_means[col])

        y_train = df_imputed.loc[train_mask, target_col]

        # Données à imputer
        X_pred = df_imputed.loc[missing_mask, predictor_cols].copy()

        # 🔧 CORRECTION: Remplir les NaN avec les moyennes
        for col in predictor_cols:
            X_pred[col] = X_pred[col].fillna(col_means[col])

        if len(X_train) == 0 or len(X_pred) == 0:
            continue

        # Entraînement Random Forest
        rf = RandomForestRegressor(
            n_estimators=100,
            max_depth=10,
            random_state=42,
            n_jobs=-1
        )
        rf.fit(X_train, y_train)

        # 🔥 EXTRACTION DES PRÉDICTIONS DE TOUS LES ARBRES
        all_tree_predictions = np.array([
            tree.predict(X_pred)
            for tree in rf.estimators_
        ])  # Shape: (n_trees, n_samples_to_impute)

        # 📊 CALCUL DE σᵢ (écart-type des prédictions)
        sigma_values = np.std(all_tree_predictions, axis=0)

        # Prédiction finale (moyenne des arbres)
        final_predictions = np.mean(all_tree_predictions, axis=0)

        # Stockage des imputations et incertitudes
        for local_idx, global_idx in enumerate(X_pred.index):
            df_imputed.loc[global_idx, target_col] = final_predictions[local_idx]
            uncertainty_dict[(global_idx, target_col)] = sigma_values[local_idx]

        print(f"  {target_col}: {len(X_pred)} valeurs | σ ∈ [{sigma_values.min():.3f}, {sigma_values.max():.3f}]")

    # 🔧 CORRECTION: Vérifier qu'il n'y a plus de NaN avant de calculer les métriques
    if df_imputed.isnull().sum().sum() > 0:
        print(f"\n⚠️  Attention: {df_imputed.isnull().sum().sum()} valeurs NaN restantes")
        # Remplir les NaN restantes avec les moyennes
        for col in feature_cols:
            df_imputed[col] = df_imputed[col].fillna(col_means[col])

    # Calculer les métriques d'imputation
    original_values = df_original.values[mask_missing.values]
    imputed_values = df_imputed.values[mask_missing.values]

    # 🔧 CORRECTION: Filtrer les NaN si jamais il en reste
    valid_mask = ~(np.isnan(original_values) | np.isnan(imputed_values))
    original_values = original_values[valid_mask]
    imputed_values = imputed_values[valid_mask]

    if len(original_values) > 0:
        mae = mean_absolute_error(original_values, imputed_values)
        mse = mean_squared_error(original_values, imputed_values)
        rmse = np.sqrt(mse)

        std_original = np.std(original_values)
        nrmse = rmse / std_original if std_original > 0 else np.nan
    else:
        mae = mse = rmse = nrmse = np.nan

    imputation_metrics = {
        'mae': mae,
        'rmse': rmse,
        'nrmse': nrmse,
        'mse': mse,
        'n_imputed': mask_missing.sum().sum()
    }

    print(f"\n✓ Imputation terminée:")
    print(f"  MAE: {mae:.4f}")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  NRMSE: {nrmse:.4f}")
    print(f"  {len(uncertainty_dict)} incertitudes σᵢ calculées")

    return df_imputed, uncertainty_dict, imputation_metrics


# =============================================================================
# SECTION 3: CALCUL DU MASQUE ADAPTATIF Mᵢ
# =============================================================================

def compute_adaptive_mask(uncertainty_dict, mask_missing):
    """
    🧮 Calcule le masque adaptatif basé sur σᵢ

    Formule:
        if was_missing:
            Mᵢ = 1 - (σᵢ / max(σ))
        else:
            Mᵢ = 1.0

    Args:
        uncertainty_dict: {(row_idx, col_name): σᵢ}
        mask_missing: DataFrame des valeurs manquantes

    Returns:
        dict: {(row_idx, col_name): Mᵢ}
        float: max(σ) utilisé pour normalisation
    """
    if len(uncertainty_dict) == 0:
        return {}, 0.0

    # Normalisation globale
    max_sigma = max(uncertainty_dict.values())

    M_dict = {}

    # Pour toutes les cellules imputées
    for key, sigma_i in uncertainty_dict.items():
        # Formule simple: Mᵢ = 1 - σᵢ/max(σ)
        M_dict[key] = 1.0 - (sigma_i / max_sigma)

    return M_dict, max_sigma


# =============================================================================
# SECTION 4: ENTRAÎNEMENT DES MODÈLES SHAP
# =============================================================================

def train_shap_models(df_imputed):
    """
    Entraîne un modèle RandomForest pour chaque variable et crée les explainers SHAP

    Returns:
        tuple: (models_dict, explainers_dict)
    """
    models = {}
    explainers = {}

    feature_cols = [col for col in df_imputed.columns if col != 'target']

    for target_var in feature_cols:
        predictor_cols = [col for col in feature_cols if col != target_var]

        X_train = df_imputed[predictor_cols]
        y_train = df_imputed[target_var]

        model = RandomForestRegressor(
            n_estimators=100,
            random_state=42,
            max_depth=5
        )
        model.fit(X_train, y_train)

        models[target_var] = model
        explainers[target_var] = shap.TreeExplainer(model)

    return models, explainers


# =============================================================================
# SECTION 5: CALCUL DU LRI ADAPTATIF
# =============================================================================

def calculate_lri_adaptive(row_idx, target_var, df_imputed, mask_missing,
                          explainers, M_dict):
    """
    📊 Calcule le LRI adaptatif avec masque variable

    LRI = Σ |SHAP_i| × M_i / Σ |SHAP_i|

    où M_i varie selon l'incertitude de l'imputation

    Returns:
        tuple: (lri_score, shap_contributions_dict)
    """
    predictor_cols = [col for col in df_imputed.columns
                     if col != 'target' and col != target_var]

    X_explain = df_imputed.loc[[row_idx], predictor_cols]
    explainer = explainers[target_var]
    shap_values = explainer.shap_values(X_explain)

    shap_row = shap_values[0] if len(shap_values.shape) > 1 else shap_values

    lri_sum = 0
    total_abs_shap = 0
    contributions = {}

    for i, pred_var in enumerate(predictor_cols):
        abs_shap = abs(shap_row[i])
        total_abs_shap += abs_shap

        # Récupérer Mᵢ adaptatif
        key = (row_idx, pred_var)
        if key in M_dict:
            M_i = M_dict[key]
        else:
            M_i = 1.0  # Valeur observée

        contribution = abs_shap * M_i
        lri_sum += contribution

        contributions[pred_var] = {
            'abs_shap': abs_shap,
            'M_i': M_i,
            'contribution': contribution
        }

    lri_score = lri_sum / total_abs_shap if total_abs_shap > 0 else 0

    return lri_score, contributions


def calculate_all_lri_adaptive(df_original, df_imputed, mask_missing,
                               explainers, M_dict, uncertainty_dict):
    """
    Calcule les scores LRI adaptatifs pour toutes les valeurs imputées

    Returns:
        pd.DataFrame: Résultats LRI avec détails
    """
    lri_results = []
    feature_cols = [col for col in df_imputed.columns if col != 'target']

    for row_idx in range(len(df_imputed)):
        for var in feature_cols:
            if mask_missing.loc[row_idx, var]:
                lri_score, contributions = calculate_lri_adaptive(
                    row_idx, var, df_imputed, mask_missing, explainers, M_dict
                )

                original_value = df_original.loc[row_idx, var]
                imputed_value = df_imputed.loc[row_idx, var]

                # Récupérer σᵢ et Mᵢ
                key = (row_idx, var)
                sigma_i = uncertainty_dict.get(key, 0.0)
                M_i = M_dict.get(key, 1.0)

                lri_results.append({
                    'row_idx': row_idx,
                    'variable': var,
                    'original_value': original_value,
                    'imputed_value': imputed_value,
                    'lri_score': lri_score,
                    'sigma_i': sigma_i,
                    'M_i': M_i,
                    'abs_error': abs(original_value - imputed_value),
                    'squared_error': (original_value - imputed_value) ** 2
                })

    df_lri = pd.DataFrame(lri_results)

    return df_lri


# =============================================================================
# SECTION 6: OPTIMISATION DU SEUIL (OPTIONNEL)
# =============================================================================

def find_optimal_threshold(df_lri, n_points=101):
    """
    Trouve le seuil optimal par la méthode du coude

    Returns:
        tuple: (optimal_threshold, df_curve, optimal_metrics)
    """
    if len(df_lri) == 0:
        return None, None, None

    min_lri = df_lri['lri_score'].min()
    max_lri = df_lri['lri_score'].max()
    thresholds = np.linspace(min_lri, max_lri, n_points)

    results = []

    for t in thresholds:
        df_above = df_lri[(df_lri['lri_score'] >= t) & (~df_lri['abs_error'].isna())]
        n_total = len(df_lri[~df_lri['abs_error'].isna()])
        n_retained = len(df_above)
        coverage = (n_retained / n_total * 100) if n_total > 0 else 0

        if n_retained > 0:
            mae = df_above['abs_error'].mean()
            rmse = np.sqrt(df_above['squared_error'].mean())
        else:
            mae = rmse = np.nan

        results.append({
            'threshold': t,
            'n_retained': n_retained,
            'coverage_%': coverage,
            'mae': mae,
            'rmse': rmse
        })

    df_curve = pd.DataFrame(results)
    df_valid = df_curve.dropna(subset=['mae', 'coverage_%']).copy()

    if len(df_valid) < 3:
        if len(df_valid) > 0:
            best_idx = df_valid['mae'].idxmin()
            optimal_threshold = df_valid.loc[best_idx, 'threshold']
            optimal_metrics = df_valid.loc[best_idx].to_dict()
        else:
            return None, df_curve, None
    else:
        # Méthode du coude
        thresholds_arr = df_valid['threshold'].values
        errors = df_valid['mae'].values

        t_norm = (thresholds_arr - thresholds_arr.min()) / (thresholds_arr.max() - thresholds_arr.min() + 1e-10)
        e_norm = (errors - errors.min()) / (errors.max() - errors.min() + 1e-10)

        p1 = np.array([t_norm[0], e_norm[0]])
        p2 = np.array([t_norm[-1], e_norm[-1]])
        line_vec = p2 - p1
        line_len = np.linalg.norm(line_vec)

        if line_len > 0:
            line_unitvec = line_vec / line_len
            distances = []

            for i in range(len(t_norm)):
                point = np.array([t_norm[i], e_norm[i]])
                vec_to_point = point - p1
                proj_length = np.dot(vec_to_point, line_unitvec)
                proj = p1 + proj_length * line_unitvec
                distance = np.linalg.norm(point - proj)
                distances.append(distance)

            elbow_idx = np.argmax(distances)
            optimal_threshold = thresholds_arr[elbow_idx]
            optimal_metrics = df_valid.iloc[elbow_idx].to_dict()
        else:
            optimal_threshold = thresholds_arr[len(thresholds_arr)//2]
            optimal_metrics = df_valid.iloc[len(df_valid)//2].to_dict()

    return optimal_threshold, df_curve, optimal_metrics


# =============================================================================
# SECTION 7: ANALYSE POUR UN MISSING RATE
# =============================================================================

def run_lri_analysis_for_missing_rate(df_original, missing_rate, random_state=42):
    """
    Exécute l'analyse LRI complète pour un missing rate

    Returns:
        dict: Résultats complets
    """
    print(f"\n{'='*80}")
    print(f"ANALYSE POUR MISSING RATE = {missing_rate*100:.0f}%")
    print(f"{'='*80}")

    # 1. Introduire des valeurs manquantes
    df_missing, mask_missing = introduce_missing_values(
        df_original, missing_rate, random_state
    )
    print(f"Valeurs manquantes introduites: {mask_missing.sum().sum()}")

    # 2. Imputation avec extraction de σᵢ
    df_imputed, uncertainty_dict, imputation_metrics = impute_with_uncertainty_extraction(
        df_missing, df_original, mask_missing
    )

    # 3. Calculer le masque adaptatif
    print(f"\nCalcul du masque adaptatif...")
    M_dict, max_sigma = compute_adaptive_mask(uncertainty_dict, mask_missing)
    print(f"  max(σ) = {max_sigma:.4f}")
    if len(M_dict) > 0:
        M_values = list(M_dict.values())
        print(f"  Mᵢ ∈ [{min(M_values):.4f}, {max(M_values):.4f}]")
        print(f"  Mᵢ moyen = {np.mean(M_values):.4f}")

    # 4. Entraîner les modèles SHAP
    print(f"\nEntraînement des modèles SHAP...")
    models, explainers = train_shap_models(df_imputed)
    print(f"  ✓ {len(models)} modèles entraînés")

    # 5. Calculer les scores LRI
    print(f"\nCalcul des scores LRI...")
    df_lri = calculate_all_lri_adaptive(
        df_original, df_imputed, mask_missing, explainers, M_dict, uncertainty_dict
    )

    if len(df_lri) > 0:
        print(f"  {len(df_lri)} valeurs analysées")
        print(f"  LRI moyen: {df_lri['lri_score'].mean():.4f}")
        print(f"  LRI médian: {df_lri['lri_score'].median():.4f}")

    # 6. Trouver le seuil optimal
    print(f"\nRecherche du seuil optimal...")
    optimal_threshold, df_curve, optimal_metrics = find_optimal_threshold(df_lri)

    if optimal_threshold is not None:
        print(f"  Seuil optimal: {optimal_threshold:.4f}")
        print(f"  Couverture: {optimal_metrics['coverage_%']:.2f}%")
        print(f"  MAE optimal: {optimal_metrics['mae']:.4f}")

    # 7. Statistiques LRI
    lri_stats = {}
    if len(df_lri) > 0:
        if optimal_threshold is not None:
            df_lri['is_reliable'] = df_lri['lri_score'] >= optimal_threshold
            n_reliable = df_lri['is_reliable'].sum()
            n_unreliable = (~df_lri['is_reliable']).sum()
        else:
            n_reliable = 0
            n_unreliable = len(df_lri)

        lri_stats = {
            'lri_mean': df_lri['lri_score'].mean(),
            'lri_median': df_lri['lri_score'].median(),
            'lri_std': df_lri['lri_score'].std(),
            'lri_min': df_lri['lri_score'].min(),
            'lri_max': df_lri['lri_score'].max(),
            'M_i_mean': df_lri['M_i'].mean(),
            'M_i_std': df_lri['M_i'].std(),
            'M_i_min': df_lri['M_i'].min(),
            'M_i_max': df_lri['M_i'].max(),
            'sigma_mean': df_lri['sigma_i'].mean(),
            'sigma_std': df_lri['sigma_i'].std(),
            'n_reliable': n_reliable,
            'n_unreliable': n_unreliable,
            'pct_reliable': (n_reliable / len(df_lri) * 100) if len(df_lri) > 0 else 0
        }

    results = {
        'missing_rate': missing_rate,
        'n_missing': mask_missing.sum().sum(),
        'imputation_metrics': imputation_metrics,
        'max_sigma': max_sigma,
        'optimal_threshold': optimal_threshold,
        'optimal_metrics': optimal_metrics,
        'lri_stats': lri_stats,
        'df_lri': df_lri,
        'df_curve': df_curve
    }

    return results


# =============================================================================
# SECTION 8: ANALYSE MULTI-SCENARIOS
# =============================================================================

def run_multiple_missing_rates(df_original, missing_rates, random_state=42):
    """
    Exécute l'analyse pour plusieurs missing rates

    Returns:
        dict: {missing_rate: results}
    """
    all_results = {}

    for missing_rate in missing_rates:
        results = run_lri_analysis_for_missing_rate(
            df_original, missing_rate, random_state
        )
        all_results[missing_rate] = results

    return all_results


def create_summary_table(all_results):
    """
    Crée un tableau récapitulatif des résultats

    Returns:
        pd.DataFrame: Tableau récapitulatif
    """
    summary_data = []

    for missing_rate, results in sorted(all_results.items()):
        imp_metrics = results['imputation_metrics']
        opt_metrics = results['optimal_metrics']
        lri_stats = results['lri_stats']
        max_sigma = results['max_sigma']

        row = {
            'Missing_Rate_%': missing_rate * 100,
            'N_Missing': results['n_missing'],
            'Max_Sigma': max_sigma,
            'Imputation_MAE': imp_metrics['mae'],
            'Imputation_RMSE': imp_metrics['rmse'],
            'Imputation_NRMSE': imp_metrics['nrmse'],
            'Optimal_Threshold': results['optimal_threshold'],
            'MAE_at_Threshold': opt_metrics['mae'] if opt_metrics else np.nan,
            'Coverage_%': opt_metrics['coverage_%'] if opt_metrics else np.nan,
            'LRI_Mean': lri_stats.get('lri_mean', np.nan),
            'LRI_Median': lri_stats.get('lri_median', np.nan),
            'LRI_Std': lri_stats.get('lri_std', np.nan),
            'M_i_Mean': lri_stats.get('M_i_mean', np.nan),
            'M_i_Std': lri_stats.get('M_i_std', np.nan),
            'M_i_Min': lri_stats.get('M_i_min', np.nan),
            'M_i_Max': lri_stats.get('M_i_max', np.nan),
            'Sigma_Mean': lri_stats.get('sigma_mean', np.nan),
            'Sigma_Std': lri_stats.get('sigma_std', np.nan),
            'N_Reliable': lri_stats.get('n_reliable', 0),
            'N_Unreliable': lri_stats.get('n_unreliable', 0),
            'Pct_Reliable': lri_stats.get('pct_reliable', 0)
        }
        summary_data.append(row)

    df_summary = pd.DataFrame(summary_data)

    return df_summary


# =============================================================================
# SECTION 9: VISUALISATIONS
# =============================================================================

def visualize_multi_scenario_results(all_results, df_summary, output_dir):
    """
    Crée des visualisations complètes pour montrer l'impact du masque adaptatif
    """
    fig = plt.figure(figsize=(20, 12))
    gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

    # 1. MAE et RMSE vs Missing Rate
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.plot(df_summary['Missing_Rate_%'], df_summary['Imputation_MAE'],
            'o-', label='MAE', linewidth=2, markersize=8, color='#2E86AB')
    ax1.plot(df_summary['Missing_Rate_%'], df_summary['Imputation_RMSE'],
            's-', label='RMSE', linewidth=2, markersize=8, color='#A23B72')
    ax1.set_xlabel('Missing Rate (%)', fontsize=11, fontweight='bold')
    ax1.set_ylabel('Erreur', fontsize=11, fontweight='bold')
    ax1.set_title('Qualité d\'Imputation vs Missing Rate', fontsize=12, fontweight='bold')
    ax1.legend(fontsize=10)
    ax1.grid(alpha=0.3)

    # 2. max(σ) vs Missing Rate
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.plot(df_summary['Missing_Rate_%'], df_summary['Max_Sigma'],
            'D-', color='#E63946', linewidth=2, markersize=8)
    ax2.set_xlabel('Missing Rate (%)', fontsize=11, fontweight='bold')
    ax2.set_ylabel('max(σ)', fontsize=11, fontweight='bold')
    ax2.set_title('Incertitude Maximale vs Missing Rate', fontsize=12, fontweight='bold')
    ax2.grid(alpha=0.3)

    # 3. LRI moyen/médian vs Missing Rate
    ax3 = fig.add_subplot(gs[0, 2])
    ax3.plot(df_summary['Missing_Rate_%'], df_summary['LRI_Mean'],
            'o-', label='LRI Moyen', linewidth=2, markersize=8, color='#06A77D')
    ax3.plot(df_summary['Missing_Rate_%'], df_summary['LRI_Median'],
            's-', label='LRI Médian', linewidth=2, markersize=8, color='#D4A5A5')
    ax3.set_xlabel('Missing Rate (%)', fontsize=11, fontweight='bold')
    ax3.set_ylabel('Score LRI', fontsize=11, fontweight='bold')
    ax3.set_title('Scores LRI vs Missing Rate', fontsize=12, fontweight='bold')
    ax3.legend(fontsize=10)
    ax3.grid(alpha=0.3)
    ax3.set_ylim([0, 1.05])

    # 4. Mᵢ moyen vs Missing Rate
    ax4 = fig.add_subplot(gs[1, 0])
    ax4.plot(df_summary['Missing_Rate_%'], df_summary['M_i_Mean'],
            'o-', color='#F18F01', linewidth=2, markersize=8)
    ax4.fill_between(df_summary['Missing_Rate_%'],
                     df_summary['M_i_Min'],
                     df_summary['M_i_Max'],
                     alpha=0.2, color='#F18F01')
    ax4.set_xlabel('Missing Rate (%)', fontsize=11, fontweight='bold')
    ax4.set_ylabel('Mᵢ', fontsize=11, fontweight='bold')
    ax4.set_title('Masque Adaptatif Moyen (avec min/max)', fontsize=12, fontweight='bold')
    ax4.grid(alpha=0.3)
    ax4.set_ylim([0, 1.05])

    # 5. σ moyen vs Missing Rate
    ax5 = fig.add_subplot(gs[1, 1])
    ax5.plot(df_summary['Missing_Rate_%'], df_summary['Sigma_Mean'],
            'o-', color='#6A4C93', linewidth=2, markersize=8)
    ax5.set_xlabel('Missing Rate (%)', fontsize=11, fontweight='bold')
    ax5.set_ylabel('σ moyen', fontsize=11, fontweight='bold')
    ax5.set_title('Incertitude Moyenne vs Missing Rate', fontsize=12, fontweight='bold')
    ax5.grid(alpha=0.3)

    # 6. Pourcentage de valeurs fiables
    ax6 = fig.add_subplot(gs[1, 2])
    ax6.plot(df_summary['Missing_Rate_%'], df_summary['Pct_Reliable'],
            'o-', color='#06A77D', linewidth=2, markersize=8)
    ax6.set_xlabel('Missing Rate (%)', fontsize=11, fontweight='bold')
    ax6.set_ylabel('% Valeurs Fiables', fontsize=11, fontweight='bold')
    ax6.set_title('Taux de Fiabilité vs Missing Rate', fontsize=12, fontweight='bold')
    ax6.grid(alpha=0.3)
    ax6.set_ylim([0, 105])

    # 7. Distribution des LRI (boxplot)
    ax7 = fig.add_subplot(gs[2, :2])
    lri_data = []
    labels = []
    for missing_rate in sorted(all_results.keys()):
        df_lri = all_results[missing_rate]['df_lri']
        if len(df_lri) > 0:
            lri_data.append(df_lri['lri_score'].values)
            labels.append(f"{missing_rate*100:.0f}%")

    if lri_data:
        bp = ax7.boxplot(lri_data, labels=labels, patch_artist=True)
        for patch in bp['boxes']:
            patch.set_facecolor('#A8DADC')
            patch.set_edgecolor('#1D3557')
        for whisker in bp['whiskers']:
            whisker.set(color='#1D3557', linewidth=1.5)
        for cap in bp['caps']:
            cap.set(color='#1D3557', linewidth=1.5)
        for median in bp['medians']:
            median.set(color='#E63946', linewidth=2)

        ax7.set_xlabel('Missing Rate', fontsize=11, fontweight='bold')
        ax7.set_ylabel('Score LRI', fontsize=11, fontweight='bold')
        ax7.set_title('Distribution des Scores LRI par Missing Rate',
                     fontsize=12, fontweight='bold')
        ax7.grid(alpha=0.3, axis='y')
        ax7.set_ylim([0, 1.05])

    # 8. Distribution des Mᵢ (boxplot)
    ax8 = fig.add_subplot(gs[2, 2])
    M_data = []
    labels2 = []
    for missing_rate in sorted(all_results.keys()):
        df_lri = all_results[missing_rate]['df_lri']
        if len(df_lri) > 0:
            M_data.append(df_lri['M_i'].values)
            labels2.append(f"{missing_rate*100:.0f}%")

    if M_data:
        bp2 = ax8.boxplot(M_data, labels=labels2, patch_artist=True)
        for patch in bp2['boxes']:
            patch.set_facecolor('#FFE5B4')
            patch.set_edgecolor('#F18F01')
        for whisker in bp2['whiskers']:
            whisker.set(color='#F18F01', linewidth=1.5)
        for cap in bp2['caps']:
            cap.set(color='#F18F01', linewidth=1.5)
        for median in bp2['medians']:
            median.set(color='#E63946', linewidth=2)

        ax8.set_xlabel('Missing Rate', fontsize=11, fontweight='bold')
        ax8.set_ylabel('Mᵢ', fontsize=11, fontweight='bold')
        ax8.set_title('Distribution des Masques Adaptatifs',
                     fontsize=12, fontweight='bold')
        ax8.grid(alpha=0.3, axis='y')
        ax8.set_ylim([0, 1.05])

    plt.suptitle('LRI ADAPTATIF AVEC σᵢ - ANALYSE MULTI-SCÉNARIOS',
                fontsize=16, fontweight='bold', y=0.998)

    # Sauvegarder
    output_file = os.path.join(output_dir, f'lri_sigma_analysis_{timestamp}.png')
    fig.savefig(output_file, dpi=300, bbox_inches='tight')
    print(f"\n✓ Graphique sauvegardé: {output_file}")
    plt.close()

    return output_file

In [ ]:
# =============================================================================
# SECTION 10: SCRIPT PRINCIPAL
# =============================================================================

if __name__ == "__main__":

    # Démarrer le logger
    logger = Logger(LOG_FILE)
    sys.stdout = logger

    try:
        print("\n" + "="*80)
        print("ANALYSE LRI ADAPTATIF AVEC σᵢ - DATASET DIABETES")
        print("="*80)
        print(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"Répertoire de sortie: {OUTPUT_DIR}")

        print("\n" + "="*80)
        print("FORMULE DU MASQUE ADAPTATIF")
        print("="*80)
        print("  if was_missing:")
        print("      Mᵢ = 1 - (σᵢ / max(σ))")
        print("  else:")
        print("      Mᵢ = 1.0")
        print("")
        print("  où σᵢ = std(prédictions des arbres)")
        print("="*80)

        # Charger le dataset
        print("\n" + "-"*80)
        print("CHARGEMENT DU DATASET")
        print("-"*80)
        df_original = load_diabetes_dataset()
        print(f"✓ Dataset Diabetes chargé")
        print(f"  Dimensions: {df_original.shape[0]} lignes × {df_original.shape[1]} colonnes")
        print(f"  Variables: {', '.join(df_original.columns.tolist())}")

        # Configuration des tests
        missing_rates = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]

        print(f"\n✓ Missing rates à tester: {[f'{mr*100:.0f}%' for mr in missing_rates]}")

        # Exécuter l'analyse
        print("\n" + "="*80)
        print("DÉMARRAGE DE L'ANALYSE")
        print("="*80)

        all_results = run_multiple_missing_rates(
            df_original, missing_rates, random_state=42
        )

        # Créer le tableau récapitulatif
        print("\n" + "="*80)
        print("CRÉATION DU TABLEAU RÉCAPITULATIF")
        print("="*80)

        df_summary = create_summary_table(all_results)

        # Afficher le tableau
        print("\n" + "="*80)
        print("TABLEAU RÉCAPITULATIF FINAL")
        print("="*80)
        print(df_summary.to_string(index=False))

        # Sauvegarder le tableau
        csv_file = os.path.join(OUTPUT_DIR, f'lri_sigma_summary_{timestamp}.csv')
        df_summary.to_csv(csv_file, index=False)
        print(f"\n✓ Tableau CSV sauvegardé: {csv_file}")

        excel_file = os.path.join(OUTPUT_DIR, f'lri_sigma_summary_{timestamp}.xlsx')
        df_summary.to_excel(excel_file, index=False, sheet_name='Summary')
        print(f"✓ Tableau Excel sauvegardé: {excel_file}")

        # Créer les visualisations
        print("\n" + "-"*80)
        print("CRÉATION DES VISUALISATIONS")
        print("-"*80)

        viz_file = visualize_multi_scenario_results(all_results, df_summary, OUTPUT_DIR)

        # Sauvegarder les résultats détaillés
        print("\n" + "-"*80)
        print("SAUVEGARDE DES RÉSULTATS DÉTAILLÉS")
        print("-"*80)

        for missing_rate, results in all_results.items():
            df_lri = results['df_lri']
            if len(df_lri) > 0:
                detail_file = os.path.join(
                    OUTPUT_DIR,
                    f'lri_details_{int(missing_rate*100)}pct_{timestamp}.csv'
                )
                df_lri.to_csv(detail_file, index=False)
                print(f"✓ Détails pour {missing_rate*100:.0f}%: {detail_file}")

        # Statistiques clés
        print("\n" + "="*80)
        print("STATISTIQUES CLÉS")
        print("="*80)

        print(f"\n{'Meilleur taux (MAE le plus bas)':.<50}")
        best_mae_idx = df_summary['Imputation_MAE'].idxmin()
        print(f"  Missing Rate: {df_summary.loc[best_mae_idx, 'Missing_Rate_%']:.0f}%")
        print(f"  MAE: {df_summary.loc[best_mae_idx, 'Imputation_MAE']:.4f}")
        print(f"  LRI moyen: {df_summary.loc[best_mae_idx, 'LRI_Mean']:.4f}")
        print(f"  Mᵢ moyen: {df_summary.loc[best_mae_idx, 'M_i_Mean']:.4f}")

        print(f"\n{'Pire taux (MAE le plus élevé)':.<50}")
        worst_mae_idx = df_summary['Imputation_MAE'].idxmax()
        print(f"  Missing Rate: {df_summary.loc[worst_mae_idx, 'Missing_Rate_%']:.0f}%")
        print(f"  MAE: {df_summary.loc[worst_mae_idx, 'Imputation_MAE']:.4f}")
        print(f"  LRI moyen: {df_summary.loc[worst_mae_idx, 'LRI_Mean']:.4f}")
        print(f"  Mᵢ moyen: {df_summary.loc[worst_mae_idx, 'M_i_Mean']:.4f}")

        print(f"\n{'Évolution du masque adaptatif':.<50}")
        print(f"  À 5%:  Mᵢ moyen = {df_summary.loc[0, 'M_i_Mean']:.4f}")
        print(f"  À 50%: Mᵢ moyen = {df_summary.loc[len(df_summary)-1, 'M_i_Mean']:.4f}")
        print(f"  Variation: {df_summary.loc[len(df_summary)-1, 'M_i_Mean'] - df_summary.loc[0, 'M_i_Mean']:.4f}")

        print(f"\n{'Évolution de lincertitude':.<50}")

        print(f"  À 5%:  max(σ) = {df_summary.loc[0, 'Max_Sigma']:.4f}")
        print(f"  À 50%: max(σ) = {df_summary.loc[len(df_summary)-1, 'Max_Sigma']:.4f}")

        # Résumé final
        print("\n" + "="*80)
        print("FICHIERS GÉNÉRÉS")
        print("="*80)
        print(f"✓ Graphique principal: {viz_file}")
        print(f"✓ Tableau CSV: {csv_file}")
        print(f"✓ Tableau Excel: {excel_file}")
        print(f"✓ Log complet: {LOG_FILE}")
        print(f"✓ Tous les fichiers dans: {OUTPUT_DIR}")

        print("\n" + "="*80)
        print("✓ ANALYSE TERMINÉE AVEC SUCCÈS")
        print("="*80)

    except Exception as e:
        print(f"\n{'='*80}")
        print(f"✗ ERREUR LORS DE L'EXÉCUTION")
        print(f"{'='*80}")
        print(f"Message d'erreur: {str(e)}")
        import traceback
        print("\nTraceback complet:")
        traceback.print_exc()

    finally:
        # Fermer le logger
        sys.stdout = logger.terminal
        logger.close()
        print(f"\n✓ Log sauvegardé dans: {LOG_FILE}")